# DINGO: fit and inspect real sources

Use the **grism** kernel. Set your input directory and source IDs, then run the cells in order: inspect inputs → fit/reopen morphology → inspect candidates → choose starting solutions → sample and save.

No earlier fits are required. A new source gets **one PSF-convolved Sérsic fit with a constant sky**, saved under its shared `morph_fits` directory and reused across kinematic experiments. The kinematic model jointly fits R+C with intrinsic dispersion fixed to zero; it samples deprojected $V_\infty$ and inclination by default.

Candidate selection is deliberate: **Run All pauses for your candidate choices** unless you enter them in `CANDIDATE_IDS`. Sources run sequentially; an ordinary error is printed and recorded in `errors`, while other sources continue. Interrupting execution stops the work.

For saved results only, execute **Settings** and the final **Reopen saved results** cell. [Full workflow guide](docs/workflow.md).

## Settings

`DATA_DIR/ID<id>/` must contain `image.fits`, `R.fits`, `C.fits` and `config_kinematics.yaml`. If configurations live elsewhere, set `CONFIG_DIR`; it uses the same `ID<id>/config_kinematics.yaml` layout. The configuration supplies source coordinates and grism metadata; the three images do not replace it.

The F150W morphology fit also needs a calibrated science mosaic, uncertainty mosaic and matching PSF. Leave the optional paths as `None` to use the local survey defaults, or supply your own paths. These are observations/calibrations, not results from another fit.

`OVERWRITE=True` resets only each named **kinematic run**. Shared morphology stays intact. `REFIT_MORPHOLOGY=True` creates a new morphology revision; use a new `RUN_NUMBER` or `OVERWRITE=True` if this run already has a morphology attached. Set the refit switch back to `False` after that run.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display
from dingo import workflow as wf

DINGO_DIR = Path(wf.__file__).resolve().parents[1]
KINEMATICS_DIR = DINGO_DIR.parent
DATA_DIR = DINGO_DIR / "data"  # Edit: root containing ID<id>/{image,R,C}.fits.
CONFIG_DIR = None              # Optional separate root containing ID<id>/config_kinematics.yaml.
WORKING_DIR = KINEMATICS_DIR / "field_kinematics" / "edr_kinematics" / "PaA" / "dingo_runs"
SOURCE_IDS = [3700]            # For example: [3700, 6170, 15665].

RUN_NUMBER = 1
DEVICE = "cuda"
INCLINATION_SIGMA_DEG = 5.0    # Total Gaussian 1 sigma width, adjustable per source below.
OVERWRITE = False
REFIT_MORPHOLOGY = False
RUN_MCMC = True

# Optional per-source prepare_source settings; no separate source-specific code cells.
SOURCE_OPTIONS = {
    # 15665: dict(inclination_mode="fixed", inclination_deg=30.0,
    #             start_velocity_kms=400.0, run_number=2),
    # 6170: dict(inclination_sigma_deg=10.0),
}
CANDIDATE_IDS = {}  # e.g. {3700: 2}; leave absent to choose interactively after inspection.

MORPHOLOGY_SETTINGS = dict(
    warmup=1000, draws=2000,
    science=None, error=None, psf=None, bundle=None,  # Optional F150W input paths.
)
MCMC_SETTINGS = dict(warmup=1000, draws=1000, walkers=64, seed=42)

errors = {}
print("Sources:", SOURCE_IDS)
print("Inputs:", DATA_DIR)
print("Save location:", WORKING_DIR)

## 1. Check and display inputs

Each source receives its own run handle. The preparation step checks all required inputs before fitting. Keep the same settings to reopen an experiment; change `RUN_NUMBER` to compare different priors or starts.

In [ ]:
runs = {}
for source_id in SOURCE_IDS:
    try:
        settings = dict(
            working_dir=WORKING_DIR, data_dir=DATA_DIR,
            run_number=RUN_NUMBER, inclination_mode="imaging_prior",
            inclination_sigma_deg=INCLINATION_SIGMA_DEG,
            device=DEVICE, overwrite=OVERWRITE,
        )
        if CONFIG_DIR is not None:
            settings["config_path"] = Path(CONFIG_DIR) / f"ID{source_id}" / "config_kinematics.yaml"
        settings.update(SOURCE_OPTIONS.get(source_id, {}))
        run = wf.prepare_source(source_id, **settings)
        _ = wf.show_inputs(run)
        runs[source_id] = run
        errors.pop((source_id, "inputs"), None)
    except Exception as exc:
        errors[source_id, "inputs"] = str(exc)
        print(f"ID{source_id} — inputs failed: {exc}")

## 2. Fit or reopen the shared morphology

This displays the actual F150W fitting image, the Sérsic model and residuals, parameter summaries and the **native PySersic corner plot**. Inclination is derived from axis ratio and printed for both $q_0=0$ and $q_0=0.2$.

A matching saved fit is reused automatically; no previous fit is needed. Its posterior, plots, **`parameters.txt`** (median and 16th–84th percentiles) and **`pysersic_<ID>_info.cat`** (settings, inputs, timing and status) live under `<ID>/morph_fits/F150W/fit_001/`. An intentional refit creates another revision and preserves the earlier one.

The kinematic run records the exact morphology revision and its inclination prior. In the default mode this supplies a soft inclination prior only; the stars do not prescribe the gas morphology. Fixed or manually specified Gaussian inclination modes retain their own chosen angles.

In [ ]:
morphology = {}
for source_id, run in runs.items():
    try:
        morphology[source_id] = wf.fit_morphology(
            run, refit=REFIT_MORPHOLOGY, display=True, **MORPHOLOGY_SETTINGS,
        )
        print(f"ID{source_id} morphology: {morphology[source_id].path}")
        errors.pop((source_id, "morphology"), None)
    except Exception as exc:
        errors[source_id, "morphology"] = str(exc)
        print(f"ID{source_id} — morphology failed: {exc}")

## 3. Search and inspect every candidate

A fresh run makes its own search using the selected prior; rerunning the same experiment reuses that search. Compare candidate velocities and coherent structures in the R/C residual maps. Candidate IDs in the table match the image panels.

For fixed inclination, this exploratory search still varies inclination. The next cell previews the exact fixed-angle starting model before MCMC. Manual starting velocities also use this search for the other coordinates.

In [ ]:
candidates = {}
for source_id in morphology:
    try:
        result = wf.find_candidates(runs[source_id])
        _ = wf.show_candidates(runs[source_id])
        candidates[source_id] = result
        errors.pop((source_id, "search"), None)
    except Exception as exc:
        errors[source_id, "search"] = str(exc)
        print(f"ID{source_id} — search failed: {exc}")

## 4. Choose and preview each start

Use an **original candidate ID from its table**, not its position after sorting. Enter a blank response to skip a source for now. You can instead set `CANDIDATE_IDS` above. If you supplied a manual velocity, the selected candidate supplies the remaining starting coordinates.

The preview is the exact initial model, including any manual velocity or fixed inclination. It performs no optimization. Use another run number to compare a different start after sampling has begun.

In [ ]:
starts = {}
for source_id in candidates:
    try:
        candidate_id = CANDIDATE_IDS.get(source_id)
        if candidate_id is None:
            answer = input(f"ID{source_id}: candidate ID to start from (blank skips): ").strip()
            if not answer:
                print(f"ID{source_id}: skipped; no MCMC will run for this source.")
                continue
            candidate_id = int(answer)
        start = wf.prepare_start(runs[source_id], candidate_id=candidate_id)
        figure = start.figure()
        display(figure)
        plt.close(figure)
        starts[source_id] = start
        errors.pop((source_id, "start"), None)
    except Exception as exc:
        errors[source_id, "start"] = str(exc)
        print(f"ID{source_id} — start selection failed: {exc}")

## 5. Run MCMC, inspect and save

MCMC runs by default. `draws` is the **total retained target per walker**, excluding warmup. To extend the same run, increase only `draws`; keep warmup, walkers, seed, start and priors unchanged. The defaults are a pilot for inspection, not a guarantee of reportable uncertainties.

PyTorch evaluates the likelihood on CUDA; emcee supplies interacting ensemble walkers. Progress bars show completed steps, elapsed time and estimated time remaining. The MCMC total includes warmup; it is not multiplied by the number of walkers. Checkpoints and detailed logs remain available during the run.

Residuals use the **highest-posterior saved draw**. Corner plots and tables use pooled medians and 16th–84th percentiles, and every walker is shown. Inspect these before interpreting intervals. Saving does not certify convergence. Set `RUN_MCMC=False` to load this experiment's existing posterior in this cell; for a completely read-only review use the final cell instead.

Final files go to `<WORKING_DIR>/<ID>/<named run>/`. Default exported profiles use illustrative radii, not established observed coverage. For reportable profiles, first establish that coverage, then call `wf.save_results(runs[ID], radii_pc=observed_radii_pc)` with your justified radius array.

In [ ]:
posteriors = {}
saved_directories = {}
for source_id in starts:
    run = runs[source_id]
    try:
        if RUN_MCMC:
            posterior = wf.fit_source(run, **MCMC_SETTINGS)
        else:
            posterior = wf.load_results(run)
        posteriors[source_id] = posterior
        _ = wf.show_results(run)
        saved_directories[source_id] = wf.save_results(run)
        print(f"ID{source_id} saved: {saved_directories[source_id]}")
        print("Run information:", saved_directories[source_id] / "run_info.txt")
        errors.pop((source_id, "sampling/review"), None)
    except Exception as exc:
        errors[source_id, "sampling/review"] = str(exc)
        print(f"ID{source_id} — sampling/review failed: {exc}")

print("Saved sources:", list(saved_directories))
if errors:
    print("Stages needing attention:", errors)

## Reopen saved results — no fitting

After restarting the kernel, execute **Settings** and this cell only. Add the complete run directories printed above; an empty list does nothing. This reads exactly those runs without selecting a latest result, running PySersic, searching or starting MCMC.

In [ ]:
SAVED_RUN_DIRECTORIES = [
    # WORKING_DIR / "3700/3700_priorinc_s5_search_run01",
]

for directory in SAVED_RUN_DIRECTORIES:
    saved_run = wf.load_run(directory)
    print("Loaded:", saved_run.directory)
    _ = wf.show_results(saved_run)